# Context Caching (KV Cache) w/ Qwen2.5

## Task Description
Demonstrate how to reuse the Key-Value (KV) cache to speed up generation when the prompt shares a common prefix. This is essential for "Chat with Data" or RAG applications where the system prompt (context) is long and static.

**Focus:** We will manually manage `past_key_values` in Hugging Face `transformers` to simulate Context Caching.

## License

This notebook is licensed under the Apache License 2.0. See the [LICENSE](https://github.com/huggingface/transformers/blob/main/LICENSE) file for more details.

This notebook focuses on open-source, lightweight, and updated models, and is optimized to run on the Google Colab Free Tier.

## Learning Objectives
(Will be updated)

In [ ]:
# Check for GPU
!nvidia-smi

## 1. Installation
We need `transformers`, `accelerate`.

In [ ]:
!pip install -q transformers accelerate

## 2. Imports

In [ ]:
import torch
import time
from transformers import AutoModelForCausalLM, AutoTokenizer

## 3. Load Data
We define a long "System Prompt" that simulates a heavy context (e.g., a document or strict instructions).

In [ ]:
system_prompt = "You are a helpful AI assistant. " * 1000 + "\nAnswer the user question concisely."
user_question = "What is 2+2?"

full_prompt = system_prompt + "\nUser: " + user_question + "\nAssistant:"

## 4. Data Cleaning
Not needed.

In [ ]:
pass

## 5. Load Model
We use `Qwen/Qwen2.5-0.5B-Instruct` for speed and efficiency.

In [ ]:
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

## 6. Inference
We compare two approaches:
1. **Naive:** Process the full prompt every time.
2. **Cached:** Process the system prompt once, save the KV cache, and reuse it.

In [ ]:

# 1. Naive Approach
print("Running Naive Approach (Full Calculation)...")
inputs_naive = tokenizer(full_prompt, return_tensors="pt").to(model.device)

start_time = time.time()
output_naive = model.generate(**inputs_naive, max_new_tokens=20)
naive_time = time.time() - start_time
print(f"Naive Time: {naive_time:.4f}s")
print("Output:", tokenizer.decode(output_naive[0], skip_special_tokens=True)[-50:])

# 2. Cached Approach
print("\nRunning Cached Approach...")
# Pre-compute KV for system prompt
system_inputs = tokenizer(system_prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    system_outputs = model(**system_inputs)
    past_key_values = system_outputs.past_key_values

# Process only the new user part
# Important: We must construct inputs that follow the system prompt
user_part = "\nUser: " + user_question + "\nAssistant:"
user_inputs = tokenizer(user_part, return_tensors="pt").to(model.device)

start_time = time.time()
# Pass past_key_values to generate.
# Note: transformers handles position_ids automatically if inputs are appended,
# but we need to ensure attention_mask is correct.
# For simplicity in this demo, we rely on generate's auto-handling when past_key_values is provided.
output_cached = model.generate(
    **user_inputs,
    past_key_values=past_key_values,
    max_new_tokens=20
)
cached_time = time.time() - start_time
print(f"Cached Time: {cached_time:.4f}s")
print("Output:", tokenizer.decode(output_cached[0], skip_special_tokens=True))

results = f"Naive: {naive_time:.4f}s, Cached: {cached_time:.4f}s"


## 7. Save Results

In [ ]:
# Kitchen Sink: Save Results
import os
import numpy as np
try:
    from PIL import Image
except ImportError:
    pass
try:
    import scipy.io.wavfile as wavfile
except ImportError:
    pass

print("Saving results...")

# 1. Text / Data / Variables
with open('results.txt', 'w') as f:
    if 'results' in locals():
        f.write(str(results))
    elif 'predictions' in locals():
        f.write(str(predictions))
    elif 'prediction' in locals():
        f.write(str(prediction))
    elif 'output' in locals():
        f.write(str(output))
    elif 'features' in locals():
        f.write(str(features))
    elif 'summary' in locals():
        f.write(str(summary))
    elif 'translation' in locals():
        f.write(str(translation))
    elif 'answers' in locals():
        f.write(str(answers))
    elif 'conversation' in locals():
        f.write(str(conversation))
    elif 'generated_text' in locals():
        f.write(str(generated_text))
    elif 'transcription' in locals():
        f.write(str(transcription))
    elif 'text' in locals():
        f.write(str(text))
    elif 'triplets' in locals():
        f.write(str(triplets))
    else:
        f.write("Task completed.")
print("Saved results.txt")

# 2. Images
if 'image' in locals():
    if hasattr(image, 'save'):
        image.save('generated_image.png')
        print("Saved generated_image.png")
elif 'images' in locals() and isinstance(images, list) and len(images) > 0:
    if hasattr(images[0], 'save'):
        images[0].save('generated_image.png')
        print("Saved generated_image.png (first image)")
    # Try to save as GIF if it's a list of images (for video/3D)
    try:
        from diffusers.utils import export_to_gif
        export_to_gif(images, "generated_video.gif")
        print("Saved generated_video.gif")
    except:
        # Fallback for standard list of PIL images
        try:
            images[0].save("generated_video.gif", save_all=True, append_images=images[1:], duration=100, loop=0)
            print("Saved generated_video.gif")
        except:
            pass

# Graph Visualization (specific to Graph RAG)
if os.path.exists("graph_visualization.png"):
    import shutil
    shutil.copy("graph_visualization.png", "generated_image.png")
    print("Saved generated_image.png from graph_visualization.png")

# 3. Audio
if 'audio' in locals():
    if isinstance(audio, tuple):
        rate, data = audio
        try:
            wavfile.write('generated_audio.wav', rate, data)
            print("Saved generated_audio.wav")
        except:
            pass
    elif hasattr(audio, 'ndim'): # numpy array
        rate = save_rate if 'save_rate' in locals() else 16000
        try:
            wavfile.write('generated_audio.wav', rate, audio)
            print("Saved generated_audio.wav")
        except:
            pass
elif 'audio_data' in locals():
    rate = save_rate if 'save_rate' in locals() else 16000
    try:
        wavfile.write('generated_audio.wav', rate, audio_data)
        print("Saved generated_audio.wav")
    except:
        pass


## 8. Advanced Usage

**Teaching Tip:** Try varying the input parameters or prompts to see how the model's output changes. This helps build intuition about the model's capabilities and limitations.

In [ ]:
print("Try increasing the system prompt length to see larger speedups.")

## 9. Troubleshooting